# 03 — Deep Q-Networks (DQN)

DQN (Mnih et al., DeepMind 2013/2015) was the breakthrough that made deep RL practical. It solved 49 Atari games from raw pixels using a single architecture, demonstrating that a neural network could learn a Q-function stable enough to train on. The two key innovations — experience replay and target networks — are still present in virtually every deep RL algorithm today.

## Why naive Q-learning + neural network fails

When you replace the Q-table with a neural network, two problems emerge:

**Correlated samples**: in standard RL, consecutive transitions (s_t, a_t, r_t, s_{t+1}) are highly correlated. SGD assumes i.i.d. samples. Training on correlated sequences causes the network to chase a moving target and diverge.

**Non-stationary targets**: the TD target `r + γ max_a' Q(s', a'; θ)` uses the same network θ that we are updating. Every update changes the target, creating a feedback loop that amplifies errors.

DQN fixes both:
1. **Experience replay buffer**: store transitions in a replay buffer; sample random mini-batches for each update → breaks correlations
2. **Target network**: a copy of Q with frozen weights θ⁻ updated periodically → stabilises targets

## The DQN algorithm

```
Initialise Q-network Q(s,a;θ) and target network Q̂(s,a;θ⁻) with θ⁻ = θ
Initialise replay buffer D

For each episode:
  For each step t:
    Select a_t = argmax_a Q(s_t, a; θ) with ε-greedy exploration
    Execute a_t, observe r_t, s_{t+1}
    Store (s_t, a_t, r_t, s_{t+1}, done) in D
    Sample mini-batch B from D
    For each (s, a, r, s', done) in B:
      y = r  if done
      y = r + γ max_{a'} Q̂(s', a'; θ⁻)  otherwise
    Update θ by minimising (y - Q(s, a; θ))²
    Every C steps: θ⁻ ← θ
```

In [ ]:
# DQN on CartPole-v1
# CartPole: balance a pole on a cart by pushing left/right
# State: [cart_pos, cart_vel, pole_angle, pole_vel] — 4 continuous dimensions
# Action: push left (0) or right (1)

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
import gymnasium as gym

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Q-Network ────────────────────────────────────────────────
class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x):
        return self.net(x)

# ── Replay Buffer ─────────────────────────────────────────────
class ReplayBuffer:
    def __init__(self, capacity=10_000):
        self.buf = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buf.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(actions).to(device),
            torch.FloatTensor(rewards).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(dones).to(device),
        )

    def __len__(self): return len(self.buf)

# ── DQN Agent ─────────────────────────────────────────────────
class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 buffer_size=10_000, batch_size=64, target_update=100):
        self.action_dim = action_dim
        self.gamma = gamma
        self.batch_size = batch_size
        self.target_update = target_update
        self.steps = 0

        self.q_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net = QNetwork(state_dim, action_dim).to(device)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.q_net.parameters(), lr=lr)
        self.buffer = ReplayBuffer(buffer_size)

    def select_action(self, state, epsilon):
        if random.random() < epsilon:
            return random.randrange(self.action_dim)
        with torch.no_grad():
            q = self.q_net(torch.FloatTensor(state).to(device))
            return q.argmax().item()

    def update(self):
        if len(self.buffer) < self.batch_size:
            return None
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)

        # Current Q values
        q_values = self.q_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)

        # TD targets using frozen target network
        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1)[0]
            targets = rewards + self.gamma * max_next_q * (1 - dones)

        loss = F.mse_loss(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_net.parameters(), 10)
        self.optimizer.step()

        self.steps += 1
        if self.steps % self.target_update == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())

        return loss.item()

# ── Training loop ─────────────────────────────────────────────
env = gym.make("CartPole-v1")
state_dim = env.observation_space.shape[0]  # 4
action_dim = env.action_space.n             # 2

agent = DQNAgent(state_dim, action_dim)

n_episodes = 400
epsilon_start, epsilon_end, epsilon_decay = 1.0, 0.01, 0.995
epsilon = epsilon_start

episode_rewards = []
losses = []

for ep in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    ep_losses = []

    for _ in range(500):
        action = agent.select_action(state, epsilon)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        agent.buffer.push(state, action, reward, next_state, float(done))
        loss = agent.update()
        if loss is not None:
            ep_losses.append(loss)

        state = next_state
        total_reward += reward
        if done:
            break

    epsilon = max(epsilon_end, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)
    if ep_losses:
        losses.append(np.mean(ep_losses))

    if (ep + 1) % 50 == 0:
        avg = np.mean(episode_rewards[-50:])
        print(f"Episode {ep+1:4d} | Avg reward (last 50): {avg:6.1f} | ε={epsilon:.3f}")

env.close()

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

window = 20
smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
ax1.plot(episode_rewards, alpha=0.3, color='steelblue', label='Raw')
ax1.plot(smoothed, color='steelblue', label=f'{window}-ep avg')
ax1.axhline(195, color='green', linestyle='--', alpha=0.7, label='Solved (195)')
ax1.set_xlabel('Episode'); ax1.set_ylabel('Total Reward')
ax1.set_title('DQN on CartPole-v1', fontweight='bold')
ax1.legend()

ax2.plot(losses, color='darkorange', alpha=0.7)
ax2.set_xlabel('Update step'); ax2.set_ylabel('MSE Loss')
ax2.set_title('TD Loss During Training', fontweight='bold')

plt.tight_layout()
plt.savefig('/tmp/dqn_cartpole.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Final 50-episode average: {np.mean(episode_rewards[-50:]):.1f} (solved threshold: 195)")


## Why DQN was the breakthrough

Before DQN, it was known that combining neural networks with Q-learning was unstable. DQN's contribution was not the network architecture but the two stabilisation tricks — replay and target networks — that made training converge reliably. The 2015 Nature paper demonstrated superhuman performance on 29 of 49 Atari games, using the same architecture with no game-specific tuning.

These two tricks survive in modern algorithms:
- Replay buffers appear in SAC, TD3, and most off-policy algorithms
- Target networks (or Polyak-averaged targets) appear in SAC, DDPG, TD3

DQN itself has been superseded for most tasks by PPO and SAC, but understanding it is prerequisite for understanding its descendants.